In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib 
import matplotlib.pyplot as plt
import sklearn
import torch
import openpyxl
from torch_geometric.data import Data
import networkx as nx
from scipy.stats import spearmanr
import seaborn as sns
from IPython.display import display, Math

#### Define Dataset Paths

In [ ]:
network_file_path = "data/2024_PhosphoAtlas 2.0_updated KSP network_withHTKAMconnections.xlsx"
measurement_file_path = "data/BRAF_CDK_kinase profile_202500827_JPC2MBetAL_v01.csv"
tumor_samples_path = "data/TumorSample_normalizedKinaseActivities_2025.xlsx"

# read in new tumor dataset
df_tumor_raw = pd.read_excel(tumor_samples_path)

# df_tumor_raw.head() # needs cleaning

In [ ]:
# Clean tumor sample dataset 

# Take row 2 as the header
new_header = df_tumor_raw.iloc[2]       # row with real column names
df_tumor_samples = df_tumor_raw.iloc[3:].copy()       # data starts from row 3
df_tumor_samples.columns = new_header           # set proper column names

# Reset index and drop completely empty columns (like 'Unnamed: 1', etc.)
df_tumor_samples = df_tumor_samples.reset_index(drop=True)
df_tumor_samples = df_tumor_samples.dropna(axis=1, how="all")

df_tumor_samples = df_tumor_samples.rename(columns={"sampple name": "sample_name"})

df_tumor_samples.head()

In [ ]:
# df_measurement.head()
# df_network.head()
# df_tumor_samples.head()

## Inital Topology Analysis

#### Definitions (plain English):
- **Directed graph**: arrows go from a kinase to a substrate.
- **Node**: a gene/protein label.
- **Edge**: a connection kinase → substrate.
- **In-degree**: how many arrows point **into** a node.
- **Out-degree**: how many arrows point **out** of a node.
- **Neighborhood connectivity**: average degree of a node's neighbors (how connected its neighbors are).
- **Density**: how full the graph is: `edges / (nodes × (nodes-1))` (for directed, no self-loops).
- **Shortest path (Dijkstra)**: the fewest steps between nodes.

#### Creates a reusable function that builds the network and calculates centrality for any PhosphoAtlas version

In [ ]:
def analyze_network(pa_df, network_name):
    """Analyze a PhosphoAtlas network and compute centrality metrics"""
    
    # Build directed graph
    # Treats each row as an edge such that KINASE GENE -> SUBSTRATE GENE
    G = nx.from_pandas_edgelist(
        pa_df,
        source='KINASE GENE',
        target='SUBSTRATE GENE',
        create_using=nx.DiGraph()
    )
    
    # Compute ALL centrality metrics (including out_degree this time!)
    # in_degree[n]  = how many kinases phosphorylate this node.
    # out_degree[n] = how many substrates this kinase hits.
    # in_degree_centrality[n] = normalized in-degree (roughly in_degree / (N−1))
    in_degree = dict(G.in_degree())
    out_degree = dict(G.out_degree())
    in_degree_cent = nx.in_degree_centrality(G)
    
    # Create dataframe with metrics
    centrality_df = pd.DataFrame({
        'node': list(in_degree.keys()),
        'in_degree': list(in_degree.values()),
        'out_degree': [out_degree[n] for n in in_degree.keys()],
        'in_degree_centrality': [in_degree_cent[n] for n in in_degree.keys()]
    })
    
    print(f"\n{network_name} Network:")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")
    
    return G, centrality_df

#### Load and analyze our network 

In [ ]:
# load and check network df edges 
df_network = pd.read_excel(network_file_path)
print(f"Network df loaded: {len(df_network)} edges")

# Clean the data
df_network = df_network[['KINASE GENE', 'SUBSTRATE GENE']].dropna()

# Normalize types: force both columns to plain upper-case Python strings (was running into issue at RWR function)
df_network["KINASE GENE"] = df_network["KINASE GENE"].astype(str).str.strip().str.upper()
df_network["SUBSTRATE GENE"] = df_network["SUBSTRATE GENE"].astype(str).str.strip().str.upper()

# Analyze network 
G_network, centrality_network = analyze_network(df_network, "2024_PhosphoAtlas 2.0_updated KSP network")

#### Load Measurement Data and Check Overlap. This is important because we can only analyze kinases that exist in both the network and your measurements.

In [ ]:
# Load measurement data and check its shape
measurement_df = pd.read_csv(measurement_file_path)
print(f"Loaded measurements: {measurement_df.shape[0]} rows, {measurement_df.shape[1]} columns")

# Get the kinase column (it might have a complex name)
kinase_col = measurement_df.columns[0]  # First column is usually the kinase names
print(f"Kinase column: {kinase_col}")

# Isolate the nodes 
network_nodes = set(G_network.nodes())

# Clean kinase names
measurement_df['kinases'] = measurement_df[kinase_col].str.strip().str.upper()
measured_kinases = set(measurement_df['kinases'].dropna())
print(f"Number of measured kinases: {len(measured_kinases)}")

print(f"Kinases only in measurements: {len(measured_kinases - network_nodes)}")

#### Kinase Name Mapping
Standardized kinase names to match PhosphoAtlas database format:

* Mappings Applied:
*   "NME" → NME1 (ambiguous name to specific kinase)
*   "NME1-NME2" → NME1 (fusion notation to primary kinase)
*   "THAP12" → PRKRIR (gene symbol to protein name used in PhosphoAtlas) Verified on PhosphositePlus and Uniprot
*   "NME2" → kept as NME2 (already correct)
 Result: 192 rows with 191 unique kinases (NME1 appears twice after mapping), achieving 100% coverage with PA2 network and 95% with PA1 network.

In [ ]:
# Checking if the kinase name mapping was already applied to meas. data 

#"NME" in measurement_df["Kinases"].values
#"NME1" in measurement_df["Kinases"].values
#"NME1-NME2" in measurement_df["Kinases"].values
#"THAP12" in measurement_df["Kinases"].values

In [ ]:
# Load measurement data
measurement_df = pd.read_csv(measurement_file_path)
print(f"Loaded measurements: {measurement_df.shape[0]} rows, {measurement_df.shape[1]} columns")

# Fix kinase column name if needed
kinase_col = measurement_df.columns[0]  
measurement_df['kinases'] = measurement_df[kinase_col].str.strip().str.upper()

# Apply all name fixes
kinase_name_mapping = {
    'NME': 'NME1',           # Ambiguous to specific
    'NME1-NME2': 'NME1',     # Fusion to primary kinase
    'THAP12': 'PRKRIR'       # Gene symbol to protein name in PA
}

measurement_df['kinases'] = measurement_df['kinases'].replace(kinase_name_mapping)

# Final results
measured_kinases = set(measurement_df['kinases'].dropna())

print(f"\nAfter standardization:")
print(f"Total rows: {len(measurement_df)}")
print(f"Unique kinases: {len(measured_kinases)}")


#### Merge Network Metrics with Experimental Measurements

In [ ]:
# === MERGE NETWORK CENTRALITY WITH EXPERIMENTAL DATA ===
# Goal: Combine network topology metrics with your kinase activity measurements
# This allows us to test if network-important kinases show different experimental behavior

# Merge PA1 centrality with measurements
merged_network = measurement_df.merge(
    centrality_network,
    left_on='kinases',
    right_on='node',
    how='inner'  # Only keep kinases present in both datasets
)

# keeps all the meas. data and topology metrics for the 187 kinases w/ measurements
merged_network.head()

In [ ]:
def plot_data_table(df, sort_by="out_degree", n=10, figsize=(11, 4)):
    """
    Render a slide-ready table showing a subset of the merged network data.

    Columns shown: kinase name, CD score, in-degree, out-degree, in-degree centrality.
    """
    cols = {
        "kinases": "Kinase",
        "Contrast_Difference": "CD Score",
        "in_degree": "In-Degree",
        "out_degree": "Out-Degree",
        "in_degree_centrality": "In-Degree\nCentrality",
    }

    subset = (
        df[list(cols.keys())]
        .sort_values(sort_by, ascending=False)
        .head(n)
        .rename(columns=cols)
        .reset_index(drop=True)
    )

    # Round floats
    subset["CD Score"] = subset["CD Score"].round(2)
    subset["In-Degree\nCentrality"] = subset["In-Degree\nCentrality"].round(4)

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")

    tbl = ax.table(
        cellText=subset.values,
        colLabels=subset.columns,
        cellLoc="center",
        loc="center",
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 1.8)

    # Style header
    for j in range(len(subset.columns)):
        tbl[0, j].set_facecolor("#2C3E50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    # Alternating row colors
    for i in range(1, len(subset) + 1):
        color = "#EAF2FB" if i % 2 == 0 else "white"
        for j in range(len(subset.columns)):
            tbl[i, j].set_facecolor(color)

    # Highlight CD Score column (positive = green tint, negative = red tint)
    cd_col_idx = list(subset.columns).index("CD Score")
    for i in range(1, len(subset) + 1):
        val = subset["CD Score"].iloc[i - 1]
        if val > 0:
            tbl[i, cd_col_idx].set_facecolor("#D5F5E3")
        elif val < 0:
            tbl[i, cd_col_idx].set_facecolor("#FADBD8")

    ax.set_title(
        f"Top {n} kinases by {sort_by.replace('_', ' ').title()}  |  Merged Network Data",
        fontsize=13, fontweight="bold", pad=16, color="#222222"
    )

    plt.tight_layout()
    plt.show()


# Sort by out-degree to show the hub kinases
plot_data_table(merged_network, sort_by="out_degree", n=10)

# Optionally: sort by CD score to show most treatment-responsive kinases
# plot_data_table(merged_network, sort_by="Contrast_Difference", n=10)


#### Add Kinase activity (CD Scores) as Node Attributes

In [ ]:
# === ADD KINASE ACTIVITY TO NETWORK NODES ===
# Attach experimental measurements directly to network nodes in G_network
# Allows network algos to use CD score activity values 

activity_col = "Contrast_Difference"   # init our activity column 

# Build mapping: kinase name -> activity value (Contrast_Difference)
# hashmap of kinases : CD_scores
kinase_activity = (
    measurement_df
    .set_index("kinases")[activity_col]
    .to_dict()
)

# Set this as a node attribute called 'activity' on your graph
nx.set_node_attributes(G_network, kinase_activity, "activity")

# Check how many nodes actually got an activity value
nodes_with_activity = [n for n in G_network.nodes() 
                       if "activity" in G_network.nodes[n]]

print(f"Nodes with activity values: {len(nodes_with_activity)}/{G_network.number_of_nodes()} nodes")
# 186 kinases have a CD score from meas, the rest do not 

In [ ]:
# print(kinase_activity)

# genes without a CD score have no activity values 
# G_network.nodes["AP2M1"]

# Now each node has 4 node attributes (3 topology based (structural) and 1 biological activity attribute (experimental))
G_network.nodes["AKT1"]
merged_network.loc[merged_network["Kinases"] == "AKT1"]

#### Spearman Correlation Analysis 

In [ ]:
# === SPEARMAN CORRELATION: NETWORK METRICS VS TREATMENT RESPONSE ===
# Why: Test if network hubs (many connections) are more affected by ENC+PAN treatment
# If yes → drug targets key regulatory nodes; If no → drug works randomly

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Define columns to test
# Experimental meas. cols 
experimental_cols = [
    "Contrast_Difference",
    "Contrast_Difference_Zscore",
    "Ttest_pval_-Log10"
]
# Network topology columns (from centrality_network)
network_metrics = ["in_degree", "out_degree", "in_degree_centrality"]

# Store all correlation results
results = []

# Use our merged dataframe that contains both network metrics and kinase activity measurements
df = merged_network.copy()

for exp_col in experimental_cols:
    for net_col in network_metrics:

        # Drop NaN pairs (Spearman requires no missing values only valid pairs)
        valid = df[[net_col, exp_col]].dropna()

        if len(valid) >= 10:     # Require enough data points

            # calculate the Spearman correlation
            rho, pval = spearmanr(valid[net_col], valid[exp_col])

            # add the results of each pair into running list 
            results.append({
                "network_metric": net_col,
                "experimental_measure": exp_col,
                "spearman_rho": rho,
                "p_value": pval,
                "n": len(valid)
            })

# Put results in a dataframe
results_df = pd.DataFrame(results)

# FDR correction (Benjamini–Hochberg) since we ran 9 tests (3 network metrics * 3 activity measures)
#
if len(results_df) > 0:
    results_df["p_value_fdr"] = multipletests(results_df["p_value"], method="fdr_bh")[1]

# Display Results
print("="*70)
print("SPEARMAN CORRELATION RESULTS")
print("="*70)
print(results_df.round(4).to_string(index=False))

# Show significant results (FDR < 0.05)
sig = results_df[results_df["p_value_fdr"] < 0.05] if "p_value_fdr" in results_df else pd.DataFrame()

if len(sig) > 0:
    print("\n✓ SIGNIFICANT CORRELATIONS (FDR < 0.05):")
    print(sig.round(4).to_string(index=False))
else:
    print("\n✗ No significant correlations after FDR correction")

Kinases with high out-degree (those that phosphorylate many downstream substrates) show significantly stronger drug-induced activity changes (larger CD scores). This could suggest that ENC+PAN preferentially influences major regulatory hubs within the signaling network rather than isolated kinases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr

# This plot answers: Do nodes with high in-degree also tend to have high out-degree?
# In our context: Are kinases that receive many signals also the ones that send many signals?

# Use your merged network dataframe
df = merged_network.copy()

# Compute rank vectors 
xr = rankdata(df['in_degree']) # high num of kinases regulate -> low num
yr = rankdata(df['out_degree']) # high num of substrates being hit - > low num 

# Spearman correlation
rho, p = spearmanr(xr, yr, nan_policy='omit')

# Plot
fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
ax.scatter(xr, yr, s=14, alpha=0.6)

# Trend line (linear fit on ranks)
m, b = np.polyfit(xr, yr, 1)
xx = np.linspace(xr.min(), xr.max(), 200)
ax.plot(xx, m*xx + b, linestyle='--', linewidth=2)

# Label the top hubs by out-degree (kinases sending many signals)
top = df.nlargest(12, 'out_degree')
for i, row in top.iterrows():
    ax.annotate(row['kinases'],      # standardized kinase name column
                (xr[i], yr[i]),
                xytext=(4, 4),
                textcoords='offset points',
                fontsize=9)

ax.set_xlabel('Rank of In-degree')
ax.set_ylabel('Rank of Out-degree')
ax.set_title(f'Network Rank–Rank Correlation | Spearman ρ={rho:.3f}, p={p:.1e}')
ax.grid(True, linestyle=':', alpha=0.35)

plt.tight_layout()
plt.show()

# Larger Ranks -> Larger Degrees

#### Random Walk with Restart for Network data.  Random Walk with Restart (RWR) simulates how signals propagate through your kinase network. Here's what it does:
Simple Explanation:
Imagine the kinase network as a city:

Nodes = Buildings (kinases)
Edges = Roads (phosphorylation connections)
Seed nodes = The starting points (the 186 measured kinases)

**The algorithm:**

Starts at your measured kinases (seed nodes)
Randomly walks along edges to connected kinases
15% chance to "restart" - teleport back to the starting kinase
Counts visits - kinases visited often are "influenced" by your seeds

What This Tells You:
High RWR score = This kinase is heavily influenced by your measured kinases through the network
Low RWR score = This kinase is isolated or far from your measured kinases
Why It's Useful:
Unlike simple degree metrics:

Out-degree only counts direct targets (1 step)
RWR traces signal flow through the entire network (multiple steps)

**For example:**

CDK1 might directly phosphorylate 300 proteins (out-degree = 300)
But through cascading effects, it might influence 1000+ proteins (high RWR score)

In our Analysis:
We use RWR to find which kinases are most influenced by our 186 measured kinases. 

(From PA1-PA2 Notebook) Note: The results showed that simple out-degree actually correlated better with drug response than RWR, suggesting the drug affects direct targets more than downstream cascades.

In [ ]:
# === DEFINE RANDOM WALK WITH RESTART FUNCTION ===
def random_walk_with_restart(G, seed_nodes, restart_prob=0.15, max_iters=1000):
    """
    Perform Random Walk with Restart (RWR) on a directed kinase–substrate network.
    
    Parameters
    ----------
    G : networkx.DiGraph
        The directed network (kinase -> substrate).
        
    seed_nodes : list of str
        Nodes where the walk should start. In this case, measured kinases.
        
    restart_prob : float
        Probability of teleporting back to the seed at each step (default 0.15).
        Higher = stays close to the seed. Lower = explores deeper.
        
    max_iters : int
        Number of steps for EACH seed’s walk.
        (Total steps = len(seed_nodes) × max_iters)
        
    Returns
    -------
    scores : dict
        A dictionary mapping each node → its normalized RWR influence score.
        Higher score → node is frequently visited → strongly influenced by seeds.
    """
    
    # Initialize total influence scores
    scores = {node: 0.0 for node in G.nodes()}
    n_seeds = len(seed_nodes)

    # Run RWR for each measured kinase
    for seed in seed_nodes:
        if seed not in G:
            continue
        
        # Visit count for this SINGLE seed's walk
        visit_count = {node: 0 for node in G.nodes()} # init hashmap {node : count}
        current = seed
        
        # Perform a long walk of max_iters steps
        for _ in range(max_iters):
            
            # Restart (teleport back) with a probability of 0.15
            if np.random.rand() < restart_prob:
                current = seed 
            else:
                # Walk to a random out-neighbor
                neighbors = list(G.successors(current))  # successors = all outgoing neighbors of current node
                if neighbors:
                    current = np.random.choice(neighbors) # choose one of them
                else:
                    # Dead end also restart
                    current = seed
            
            visit_count[current] += 1
        
        # Normalize by number of steps and seeds
        # count / max_iters  =  proportion of time spent on this node
        # divide again my number of seeds -> contribution averaged across all seeds 
        # add to running hashmap of node and normalized RWR influence scores
        for node, count in visit_count.items():
            scores[node] += (count / max_iters) / n_seeds

    return scores

print("Random Walk with Restart function loaded.")

#### Running RWR on our Kinase Network

In [ ]:
# Seed nodes = kinases with CD measurements
seed_nodes = list(merged_network['kinases'].dropna())

print(f"Total seed nodes: {len(seed_nodes)}")

# Run RWR
rwr_scores = random_walk_with_restart(
    G_network,
    seed_nodes,
    restart_prob=0.15,
    max_iters=1000
)

# Show top 10 nodes most influenced
top10 = sorted(rwr_scores.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 influenced nodes:")
for gene, score in top10:
    print(f"{gene:10s}  {score:.4f}")

In [ ]:
def plot_rwr_overview(rwr_scores, merged_df, n=15, figsize=(10, 5)):
    """
    Horizontal bar chart of top N kinases by RWR score.
    Bars are colored by CD score: green = positive response, red = negative, gray = unmeasured.
    """
    # Build dataframe from RWR scores
    rwr_df = (
        pd.DataFrame(rwr_scores.items(), columns=["gene", "rwr_score"])
        .sort_values("rwr_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

    # Merge in CD score for coloring
    cd_map = merged_df.set_index("kinases")["Contrast_Difference"].to_dict()
    rwr_df["cd_score"] = rwr_df["gene"].map(cd_map)

    # Assign bar colors
    def bar_color(cd):
        if pd.isna(cd):   return "#AAAAAA"   # unmeasured
        elif cd > 0:      return "#2ECC71"   # positive response
        else:             return "#E74C3C"   # negative response

    colors = rwr_df["cd_score"].apply(bar_color)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(rwr_df["gene"][::-1], rwr_df["rwr_score"][::-1], color=colors[::-1], edgecolor="white")

    ax.set_xlabel("RWR Influence Score", fontsize=11)
    ax.set_title(f"Top {n} Network-Influenced Nodes  |  Random Walk with Restart",
                 fontsize=13, fontweight="bold", pad=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=10)

    # Legend
    from matplotlib.patches import Patch
    legend = [
        Patch(color="#2ECC71", label="Positive CD (more active under treatment)"),
        Patch(color="#E74C3C", label="Negative CD (less active under treatment)"),
        Patch(color="#AAAAAA", label="Not directly measured"),
    ]
    ax.legend(handles=legend, fontsize=9, loc="lower right")

    plt.tight_layout()
    plt.show()

plot_rwr_overview(rwr_scores, merged_network, n=15)


In [ ]:
influenced_nodes = ["TP53", "CDK1", "NME1", "GSK3B", "AKT1", "CREB1", "EIF2S1", "ERBB3", "TXK", "MAPT"]
# examining some of these nodes
merged_network[merged_network["kinases"].isin(influenced_nodes)]

#### Dijkstra Shortest Path 
Q: How many steps (edges) does it take for signal to travel from one measured kinase to another?

In other words:

How many phosphorylation events separate two kinases? This captures the network proximity (how close kinases are to each other through regulatory actions)


In [ ]:
# Define Dijkstra Algo. Function
def calculate_avg_shortest_paths(G, measured_kinases):
    """Compute the average shortest path from all measured kinases to each measured kinase."""
    
    # Keep only kinases that actually exist in the graph
    measured_in_network = set(measured_kinases) & set(G.nodes())
    print(f"Calculating paths among {len(measured_in_network)} measured kinases...")
    
    avg_path_lengths = {} # store: kinase -> avg shortest path length
    
    # store: kinase -> avg shortest path length
    for target in measured_in_network:
        lengths = [] # list of shortest path lengths coming into* this target
        
        # Compute path from every other measured kinase into this target
        for source in measured_in_network:
            if source == target: #skip self-to-self
                continue
            try:
                # Compute shortest path length using Dijkstra
                length = nx.shortest_path_length(G, source, target)
                lengths.append(length)
            except nx.NetworkXNoPath:
                pass # if no path exists, ignore
        
        # if no path exists, ignore
        if lengths:
            avg_path_lengths[target] = np.mean(lengths)
    
    return avg_path_lengths

In [ ]:
measured_list = list(merged_network['kinases'].dropna())

avg_paths = calculate_avg_shortest_paths(G_network, measured_list)

In [ ]:
merged_network['avg_path_length'] = merged_network['kinases'].map(avg_paths)

valid = merged_network[['avg_path_length', 'Contrast_Difference']].dropna() # clean for spearman 

rho, p = spearmanr(valid['avg_path_length'], valid['Contrast_Difference'])
print(f"Shortest-path vs CD: rho={rho:.3f}, p={p:.4f}")

**Response:**

Dijkstra shortest-path distance answers:

How many kinase–substrate hops separate one kinase from another?

But CD values measure treatment response and thats not guaranteed to correlate with:
* graph proximity
* pathway distance
* or network closeness in a directed signaling graph.

My guess is that:

Kinase signaling is not "small-world" in the same sense that social networks are. The effect of ENC+PAN is typically local (direct phosphorylation inhibition) rather than path-propagated across long chains. Although its a little strange that PA-1 had a non significant p-value and PA-2 did (but I'm assuming our network right now is similar to PA-2).

In [ ]:
import collections

distances = []
for u in seed_nodes:
    for v in seed_nodes:
        if u != v:
            try:
                distances.append(nx.shortest_path_length(G_network, u, v))
            except nx.NetworkXNoPath:
                pass

collections.Counter(distances)

# Most measured kinases are only 2–4 hops away from each other
# Since there is very little variation, Spearman correlation naturally becomes very small (hard to parse correlation)

Creating an arbitrary importance score (WIP) 

In [ ]:
# Assign avg shorted path lengths to a hashmap variable
avg_path_length_dict = calculate_avg_shortest_paths(G_network, seed_nodes)

# Add RWR scores
merged_network["rwr_score"] = merged_network["kinases"].map(rwr_scores)

# Add shortest path scores
merged_network["avg_path_length"] = merged_network["kinases"].map(avg_path_length_dict)

#### Inital Rankings

In [ ]:
# Tweak the weights here, right now its just an arbitrary linear combination 
# WIP again :( 
merged_network["importance"] = (
    0.40 * merged_network["out_degree"] +
    0.25 * merged_network["in_degree"] +
    0.20 * merged_network["rwr_score"] -
    0.15 * merged_network["avg_path_length"]
)

In [ ]:
ranked = merged_network.sort_values("importance", ascending=False)
ranked.head(20)

## Follow Ups 
- As an experiment -> use phosphorylated site (S, T ,Y) feature as another node attribute in topology analysis (experiment 1)
- then run the GNN without phosphosites w/ Tissue Samples (exp 2) and run GNN with phosphosites (exp 3)

next steps
- Conduct RWR with weights
- GNN will use the 44 Tissue Samples to clasify which are VEH vs ENC+PAN -> class label loss metric with BCE
- See how well we can classify tissue samples 
- 

#### Weighted Random Walk With Restart

**First Idea: new path probabilities with weighted edge**

* Usually in weighted RWR, we consider weighting transitions based on edge strengths (kinases choose next path based on weighted edges -> higher weights = higher probability), but our graph currently has no good edge weights only node weights (CD scores), so we cannot meaningfully bias neighbor selection yet.

**Second Idea: weighted contributions based off CD score** 
* In standard RWR, each seed kinase contributes equally because every seed gets one walk and all walks are averaged with weight = $1/n_{\text{seeds}}$.

* In this approach, we assign each seed a weight $w_s$ based on its CD score.
This means that high-CD kinases push more influence into the network, and low-CD kinases contribute less.

* In other words, after each seed’s walk, instead of adding $\frac{1}{n_{\text{seeds}}}$ of its visit counts, we scale its contribution by its normalized CD weight $w_s$.

* Hopefully this will reflect both network topology and biological strength (response) of a kinase to the ENC+PAN treatment


#### Equations behind approach

Original RWR: $\text{score}(v) = \frac{1}{|S|} \sum_{s \in S} p_s(v)$ and each seed was weighted equally

For each seed kinase called $s$

Let  $a_s = | \text{CD}_s |$ = absolute Contrast_Difference (a big change (up or down) yields a large weight)

Normalize across all seeds:
$w_s = \frac{a_s}{\sum_j a_j}$

So now $w_s$ is the weight of that seed

After each random walk from each seed, then we combine them as a weighted average score for each node (v)!

$\text{score}(v) = \sum_{s} w_s \cdot p_s(v)$

$\text{score}(v) = w_{s_1} p_{s_1}(v) + w_{s_2} p_{s_2}(v) + w_{s_3} p_{s_3}(v) + \cdots$

Here $p_s(v)$ = fraction of time walk starting at seed $s$ spends on node $v$ 

In [ ]:
def weighted_random_walk_with_restart(
    G,
    seed_scores,          # dict: seed_kinsase -> raw score (e.g. Contrast_Difference)
    restart_prob=0.15,
    max_iters=1000
):
    """
    Weighted Random Walk with Restart (RWR) on a directed kinase–substrate network.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph (KINASE GENE -> SUBSTRATE GENE).

    seed_scores : dict
        Mapping {seed_node: raw_score}, e.g. Contrast_Difference for measured kinases.
        These raw scores will be converted into non-negative weights w_s that sum to 1.

    restart_prob : float
        Probability of teleporting back to the seed at each step (default 0.15).

    max_iters : int
        Number of steps for EACH seed’s walk.
        Total number of random-walk steps = len(valid_seeds) * max_iters.

    Returns
    -------
    scores : dict
        node -> weighted RWR influence score.
        Higher score = node is frequently visited, especially from high-weight seeds.
    """

    # 1) Filter to seeds that actually exist in the graph 
    valid_seeds = [s for s in seed_scores.keys() if s in G]
    if not valid_seeds:
        raise ValueError("No seed nodes found in the graph.")

    # 2) Build normalized seed weights w_s 
    # Example: use positive part of the score so negative CDs don't get negative weights
    raw = np.array([max(0.0, float(seed_scores[s])) for s in valid_seeds], dtype=float)

    if raw.sum() == 0:
        # Fallback to uniform if everything is <= 0
        w = np.ones_like(raw) / len(raw)
    else:
        w = raw / raw.sum()   # normalize so sum_s w_s = 1

    seed2weight = {s: w_i for s, w_i in zip(valid_seeds, w)}

    # 3) Initialize total influence scores over all nodes 
    scores = {node: 0.0 for node in G.nodes()}

    # 4) Run one RWR per seed, accumulate with weights 
    for seed in valid_seeds:
        weight_s = seed2weight[seed]

        # Visit count for this seed
        visit_count = {node: 0 for node in G.nodes()}
        current = seed

        for _ in range(max_iters):

            # Restart with prob = restart_prob
            if np.random.rand() < restart_prob:
                current = seed
            else:
                neighbors = list(G.successors(current))
                if neighbors:
                    current = np.random.choice(neighbors)
                else:
                    # Dead end -> restart
                    current = seed

            visit_count[current] += 1

        # Convert visit counts to a probability distribution p_s(v)
        # p_s(v) = visit_count[v] / max_iters
        for node, count in visit_count.items():
            p_sv = count / max_iters
            scores[node] += weight_s * p_sv   # weighted contribution from this seed

    return scores

In [ ]:
# Build seed_scores from CD (or CD z-score, etc.)
# Example: use Contrast_Difference directly
seed_scores = (
    merged_network
    .set_index("kinases")["Contrast_Difference"]
    .to_dict()
)

weighted_rwr_scores = weighted_random_walk_with_restart(
    G_network,
    seed_scores=seed_scores,
    restart_prob=0.15,
    max_iters=1000
)

# Attach to merged_network
merged_network["rwr_weighted"] = merged_network["kinases"].map(weighted_rwr_scores)

# Quick look at top kinases by weighted RWR
top_w = merged_network.sort_values("rwr_weighted", ascending=False).head(20)
top_w[["kinases", "Contrast_Difference", "in_degree", "out_degree", "rwr_weighted"]]

In [ ]:
# looking at kinases with weighted influence values
weighted_influenced = [
    "PDGFRA", "MAP3K1", "GSK3B", "BRAF", "MAP3K8",
    "MAP2K1", "ARAF", "ABL2", "RPS6KB2", "FLT1"
]

merged_network[merged_network["kinases"].isin(weighted_influenced)][
    ["kinases", "Contrast_Difference", "in_degree", "out_degree", "rwr_weighted"]
].sort_values("rwr_weighted", ascending=False)

#### Results of weighted RWR

If our original RWR was trying to answer : "Which kinases are most central structurally?" 

Then our drug-response-weighted influence score trys to answer

"Which kinases are most influenced by the kinases whose activity changed the most under treatment?"

However this weighting does not automatically mean high scores are more impactful drug targets since a high CD score might just reflect a genuine drug response, tumor-specific biology, etc... 

So since the weighting formula multiplies each walk’s influence by the seed’s CD score,
seeds with high CD scores contribute disproportionately to the final RWR probability, and the nodes around those seeds gain higher scores (regardless of network connectivity)




In [ ]:
# merged_network["importance_V2"] = (
#     0.40 * merged_network["out_degree"] +
#     0.25 * merged_network["in_degree"] +
#     0.20 * merged_network["rwr_weighted"] -
#     0.15 * merged_network["avg_path_length"]
# )

#### Side note - Quick visualization function 

* Takes a kinase and shows its neighborhood (both in and out nodes)
* In other words it's ego-network = the neighborhood of a focal node (called the "ego)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def plot_ego(G, gene, radius=1, figsize=(8,8)):
    """
    Plot the ego-network (local neighborhood) of a given kinase/substrate.
    
    Parameters
    ----------
    G : networkx.DiGraph
        Your kinase–substrate network.
        
    gene : str
        Name of the kinase/substrate to visualize.
        
    radius : int
        How many hops out from the gene to include.
        radius=1 → immediate neighbors only.
        
    figsize : tuple
        Size of the plot.
    """
    
    if gene not in G:
        print(f"Gene '{gene}' not found in network.")
        return
    
    # Extract ego graph (local neighborhood)
    ego = nx.ego_graph(G, gene, radius=radius, undirected=False)
    
    # Layout for clean visualization
    pos = nx.spring_layout(ego, seed=42)
    
    # Plot
    plt.figure(figsize=figsize)

    # Highlight our central node
    node_colors = ["orange" if n == gene else "skyblue" for n in ego.nodes()]

    nx.draw(
        ego, pos,
        with_labels=True,
        node_size=800,
        font_size=9,
        node_color=node_colors,
        edge_color="gray",
        arrowsize=12
    )
    plt.title(f"Ego-network for {gene} (radius={radius})")
    plt.show()

In [ ]:
plot_ego(G_network, "BRAF", radius=2)

## Tumor Dataset (Sample Profiles - Sheet 1) Data Dictionary:

`sample_name`
* A unique identifier describing the PDX model, experiment date, treatment, and tumor replicate.

`HTKAM_Experiment_Date`
* Date the kinase assay experiment was performed.

`PDX_model_ID`
* The patient tumor model (e.g., CN1571).
	•	This stays consistent across rows so it means all tumors came from the same patient-derived model.

`PTC_Mouse_Study_ID`
* ID of the mouse experiment these tumors were a part of.

`Treatment` - (This is what we will split on for classification or regression tasks)
* The most important label for supervision.
* UNT = untreated
* ENC+PAN = drug-treated

`Tumor_ID`
* Identifier for which specific tumor piece was used

`Tumor_Piece_#`
* Indicates which piece of the tumor was assayed.

`HTKAM_Independent_Replicate_#`
* Biological replicate number.

`Tum_Vol_mm3_at_collection`
Tumor volume at time of harvest: a potential phenotypic readout of treatment response.


# GNN Approach 



#### Overview of Approach

1. **The Graph (PA2 Network) is Fixed**
	* Nodes = kinases + substrates
	* Edges = phosphorylation relationships
	* These relationship do not changes across tumors
	* This becomes the backbone graph the GNN always operates on

So the topology is constant and only node features will vary.

2. **Tumor Samples Provide Tumor-Specific Node Features**

Each tumor has its own activity profile.

Example: (Tumor A):
(AKT1 = 2.1, BRAF = −0.4, MAP3K8 = 1.7, … etc.)

Even though the graph stays the same:
* Each tumor produces a different node-feature matrix
* Missing kinases get 0 or another imputation rule

Essentially the GNN sees 44 different “views” of the same graph, each with its own biology.

3. **The GNN Learns a Mapping From Node Features -> Drug Response**

* We choose a **supervised target** such as:
	*	Tumor volume change
	*	**ENC+PAN vs UNT class label** (Treated or Untreated Sample)
	*	Or another continuous response metric

The model will learn things like:
* Which kinases matter most across tumors
* Which network neighborhoods carry predictive signal
* How activity patterns propagate through the phosphorylation graph

**Message Passing**
* This is where GNN message passing is powerful and actually comes through 
* It lets each tumor’s activity pattern diffuse through the fixed network.


4. **Training Overview**

For each tumor t:
1. Use the same edge_index
2.	Build xᵗ = node features for tumor t
3.	Feed into GNN → get prediction ŷᵗ
4.	Compute loss L(ŷᵗ, yᵗ)
5.	Backprop through the shared graph model

**Across all tumors, the GNN learns:**
* Node embeddings
* Important pathways
* Predictive subnetworks


5. **After Training: Rank Kinases by Importance**

We can extract:
* GNN feature gradients (saliency)
* Node attention scores (GAT)
* Integrated gradients
* Or embed-space distances

These produce a GNN-based kinase importance ranking, complementing:
* Out-degree
* RWR
* Weighted RWR
* Centrality


In [ ]:
# Lets examine our tumor sample set again 
df_tumor_samples.head()

In [ ]:
len(df_tumor_samples)
df_tumor_samples['Treatment'].value_counts()
# df_tumor_samples.isna().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5), dpi=120)

sns.boxplot(
    data=df_tumor_samples,
    x="Treatment",
    y="Tum_Vol_mm3_at_collection",
    palette={"UNT": "#1f77b4", "ENC+PAN": "#d62728"}
)

sns.stripplot(
    data=df_tumor_samples,
    x="Treatment",
    y="Tum_Vol_mm3_at_collection",
    color="black",
    size=4,
    alpha=0.6
)

plt.title("Tumor Volume at Collection by Treatment Group", fontsize=14)
plt.xlabel("Treatment")
plt.ylabel("Tumor Volume (mm³)")
plt.tight_layout()
plt.show()

#### Graph Set Up 
### Creating our PyG object (Data)


In [ ]:
# 1) Build node index + edge_index
# Use all nodes from our NetworkX graph
nodes = sorted(G_network.nodes())
node2idx = {g: i for i, g in enumerate(nodes)}
n_nodes = len(nodes)
print(f"# nodes: {n_nodes}")

# Build edge_index from G_network edges (kinase -> substrate)
src_idx = []
dst_idx = []
for u, v in G_network.edges():
    src_idx.append(node2idx[u])
    dst_idx.append(node2idx[v])

edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)
print("edge_index shape:", edge_index.shape)

# 2) Build node-level feature table
# Start with centrality metrics
feat_df = pd.DataFrame({"node": nodes})

# Merge in centrality (in_degree, out_degree, in_degree_centrality)
feat_df = feat_df.merge(
    centrality_network[["node", "in_degree", "out_degree", "in_degree_centrality"]],
    on="node",
    how="left"
)

# Merge in experimental + RWR info from merged_network
# merged_network has columns: 'kinases', 'Contrast_Difference', 'rwr_weighted', 'avg_path_length', ...
feat_df = feat_df.merge(
    merged_network[["kinases", "Contrast_Difference", "rwr_weighted", "avg_path_length"]],
    left_on="node",
    right_on="kinases",
    how="left"
)

# Drop duplicate 'kinases' column, we keep 'node' as the main name
feat_df = feat_df.drop(columns=["kinases"])

# is_measured = 1 if we have a CD value, else 0
feat_df["is_measured"] = (~feat_df["Contrast_Difference"].isna()).astype(float)

# Fill any remaining NaNs with 0
feat_df = feat_df.fillna(0.0)

# 3) Turn features + labels into tensors
feature_cols = [
    "in_degree",
    "out_degree",
    "in_degree_centrality",
    "Contrast_Difference",
    "rwr_weighted",
    "avg_path_length",
    "is_measured",
]

X = feat_df[feature_cols].to_numpy(dtype=np.float32)
x = torch.tensor(X, dtype=torch.float32)

# Target = CD score (we'll mask where not measured)
y = torch.tensor(feat_df["Contrast_Difference"].to_numpy(dtype=np.float32))

# Boolean mask: which nodes have a real CD measurement
train_mask = torch.tensor(feat_df["is_measured"].to_numpy(dtype=bool))

print("x shape:", x.shape)
print("y shape:", y.shape)
print("train_mask sum (measured nodes):", train_mask.sum().item())

In [ ]:
# 4) Create PyG Data object
data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
)

data.train_mask = train_mask

data